In [18]:
import sys

print(sys.version)
print(sys.executable)

3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
c:\Users\user\OneDrive\Desktop\genai_projects\.venv\Scripts\python.exe


In [19]:
# Import Pinecone client for vector database operations
from pinecone import Pinecone

# Import PDF loader to load multiple PDF files from a directory
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Import text splitter to divide documents into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Import OpenAI embeddings and LLM for vector creation and text generation
from langchain_openai import OpenAIEmbeddings, OpenAI

# Import Pinecone vector store for storing and searching embeddings
from langchain_pinecone import PineconeVectorStore

# Print a message to confirm all imports are working
print("✅ ALL IMPORTS SUCCESSFUL!")

✅ ALL IMPORTS SUCCESSFUL!


In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [20]:
import os

In [21]:
## Lets Read the document
def read_doc(directory):
    file_loader=PyPDFDirectoryLoader(directory)
    documents=file_loader.load()
    return documents

In [22]:
doc=read_doc('documents/')
len(doc)

58

In [23]:
## Divide the docs into chunks
### https://api.python.langchain.com/en/latest/text_splitter/langchain.text_splitter.RecursiveCharacterTextSplitter.html#
def chunk_data(docs,chunk_size=800,chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
    doc=text_splitter.split_documents(docs)
    return docs

In [24]:
documents=chunk_data(docs=doc)
len(documents)

58

In [40]:
from langchain_huggingface import HuggingFaceEmbeddings

# Create a local embedding model for converting text into vectors
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Generate an embedding for a query
vectors = embeddings.embed_query("How are you?")

print(len(vectors))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5168.30it/s]


384


In [45]:
## Vector Search DB In Pinecone

import os
from dotenv import load_dotenv
from pinecone import Pinecone

# Load environment variables from .env
load_dotenv()

# Initialize Pinecone client
pc = Pinecone(
    api_key=os.environ["PINECONE_API_KEY"]
)

# Define Pinecone index name
index_name = "langchainvector"

print("✅ Pinecone initialized successfully!")

✅ Pinecone initialized successfully!


In [47]:
# Display available Pinecone indexes
pc.list_indexes()

[
    {
        "name": "langchainvect",
        "metric": "cosine",
        "host": "langchainvect-9oery6u.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1024,
        "deletion_protection": "disabled",
        "tags": null,
        "embed": {
            "model": "llama-text-embed-v2",
            "field_map": {
                "text": "text"
            },
            "dimension": 1024,
            "metric": "cosine",
            "write_parameters": {
                "dimension": 1024.0,
                "input_type": "passage",
                "truncate": "END"
            },
            "read_parameters": {
                "dimension": 1024.0,
                "input_type": "query",
                "truncate": "END"
    

In [49]:
len(vectors)

384

In [51]:
from pinecone import ServerlessSpec

# Create the Pinecone index if it does not already exist
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("✅ Pinecone index is ready!")

✅ Pinecone index is ready!


In [53]:
# Connect to the Pinecone index
index = pc.Index(index_name)

print("✅ Connected to:", index_name)

✅ Connected to: langchainvector


In [ ]:
index = pc.Index(index_name) # you have your embeddings, let's replace only this portion with our current stack.

In [59]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a text splitter to divide documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=50
)

print("✅ text_splitter created")

✅ text_splitter created


In [60]:
# Split the documents into smaller chunks
chunks = text_splitter.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 140


In [61]:
from langchain_pinecone import PineconeVectorStore

# Store document chunks and their embeddings in Pinecone
vector_store = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name
)

print("✅ 140 chunks stored in Pinecone!")

✅ 140 chunks stored in Pinecone!


In [62]:
# Check the number of vectors stored in Pinecone
print(index.describe_index_stats())

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 140}},
 'total_vector_count': 140,
 'vector_type': 'dense'}


Similarity Check

In [64]:
# Search Pinecone for the most relevant chunks
def retrieve_query(query, k=2):
    
    # Find the top-k similar documents
    matching_results = vector_store.similarity_search(
        query,
        k=k
    )
    
    return matching_results

In [65]:
# Ask a question related to the PDF
our_query = "How much will the agriculture credit target be increased?"

# Retrieve the two most relevant chunks
results = retrieve_query(our_query, k=2)

# Display the retrieved chunks
for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(result.page_content)


--- Result 1 ---
7 
 
 
 
farmers in contributing to the health of fellow citizens by growing these 
‘Shree Anna’.  
22. Now to make India a global hub for ' Shree Anna', the Indian Institute 
of Millet Research, Hyderabad will be supported as the Centre of Excellence 
for sharing best practices, research and technologies at the international 
level.    
Agriculture Credit  
23. The agriculture credit target will be increased  
to ` 20 lakh crore with focus on animal husbandry, dairy and fisheries.  
Fisheries 
24. We will launch a new sub-scheme of PM Matsya Sampada Yojana 
with targeted investment of ` 6,000 crore to further enable activities of 
fishermen, fish vendors, and micro & small enterprises, improve value chain 
efficiencies, and expand the market. 
Cooperation

--- Result 2 ---
6 
 
 
 
inclusive, farmer-centric solutions through relevant information services for 
crop planning and health, improved access to farm inputs, credit, and 
insurance, help for crop estimation, m

In [66]:
import os
from langchain_groq import ChatGroq

# Create Groq LLM for generating answers
llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model="llama-3.1-8b-instant",
    temperature=0.5
)

print("✅ Groq LLM ready!")

✅ Groq LLM ready!


In [67]:
# Test the Groq LLM
response = llm.invoke("What is 2 + 2?")

print(response.content)

The answer to 2 + 2 is 4.


In [81]:
# Generate an answer using Pinecone retrieved documents and Groq
def retrieve_answers(query):

    # Retrieve the most relevant documents from Pinecone
    doc_search = retrieve_query(query, k=2)

    # Combine the retrieved document content into one context
    context = "\n\n".join(
        doc.page_content for doc in doc_search
    )

    # Create a prompt using the retrieved context and user question
    prompt = f"""
Answer the question using only the information provided in the context.

Context:
{context}

Question:
{query}

Answer:
"""

    # Send the prompt to Groq
    response = llm.invoke(prompt)

    # Return only the generated answer
    return response.content

In [82]:
# Ask a question about the PDF
our_query = "How much the agriculture target will be increased by how many crore?"

# Generate the answer using RAG
answer = retrieve_answers(our_query)

print(answer)

The agriculture credit target will be increased to ` 20 lakh crore. 

To find out the increase, we need to know the previous target. However, it is not mentioned in the given context.
